In [13]:
import cv2
import numpy as np
import random
import os

def get_all_grid_segments(grid_size=3):
    """返回所有合法的格点间线段（不含自环），输出为(indexA, indexB)形式，index=行*grid_size+列"""
    segments = []
    for i1 in range(grid_size):
        for j1 in range(grid_size):
            idx1 = i1 * grid_size + j1
            for i2 in range(grid_size):
                for j2 in range(grid_size):
                    idx2 = i2 * grid_size + j2
                    if idx1 != idx2:
                        seg = tuple(sorted([idx1, idx2]))
                        if seg not in segments:
                            segments.append(seg)
    return segments

def ij2xy(i, j, grid_size, img_size, margin):
    gap = (img_size - 2 * margin) // (grid_size - 1)
    x = margin + j * gap
    y = margin + i * gap
    return (x, y)

def generate_main_path(grid_size=3, path_len=6):
    idx = random.randint(0, grid_size*grid_size-1)
    path = [idx]
    for _ in range(path_len-1):
        i, j = divmod(path[-1], grid_size)
        neighbors = []
        for di in [-1, 0, 1]:
            for dj in [-1, 0, 1]:
                if di == 0 and dj == 0: continue
                ni, nj = i+di, j+dj
                if 0 <= ni < grid_size and 0 <= nj < grid_size:
                    neighbors.append(ni*grid_size + nj)
        path.append(random.choice(neighbors))
    return path

def path_to_segments(path, grid_size):
    segs = set()
    for i in range(len(path)-1):
        a, b = path[i], path[i+1]
        seg = tuple(sorted([a, b]))
        segs.add(seg)
    return segs

def draw_segments(img, segs, grid_size, img_size, margin, color, thickness):
    for a, b in segs:
        i1, j1 = divmod(a, grid_size)
        i2, j2 = divmod(b, grid_size)
        pt1 = ij2xy(i1, j1, grid_size, img_size, margin)
        pt2 = ij2xy(i2, j2, grid_size, img_size, margin)
        cv2.line(img, pt1, pt2, color, thickness, lineType=cv2.LINE_AA)

def erase_main_line(img, main_segs, grid_size, img_size, margin):
    if not main_segs: return main_segs
    main_segs = set(main_segs)
    seg = random.choice(list(main_segs))
    main_segs.remove(seg)
    i1, j1 = divmod(seg[0], grid_size)
    i2, j2 = divmod(seg[1], grid_size)
    pt1 = ij2xy(i1, j1, grid_size, img_size, margin)
    pt2 = ij2xy(i2, j2, grid_size, img_size, margin)
    cv2.line(img, pt1, pt2, (255,255,255), 16, lineType=cv2.LINE_AA)
    return main_segs

def add_noise_segments(all_segments, main_segs, n_noise=3):
    candidates = [s for s in all_segments if s not in main_segs]
    chosen = set()
    segs = []
    while len(segs) < n_noise and candidates:
        s = random.choice(candidates)
        if s not in chosen and s[::-1] not in chosen:
            segs.append(s)
            chosen.add(s)
    return segs

def generate_one(n, all_segments, img_dir, grid_size=3, img_size=400, margin=60, path_len=None):
    if path_len is None:
        path_len = random.randint(4,7)
    main_path = generate_main_path(grid_size, path_len)
    main_segs = path_to_segments(main_path, grid_size)

    # 保存主图
    img = np.ones((img_size, img_size, 3), np.uint8) * 255
    draw_segments(img, main_segs, grid_size, img_size, margin, (0,0,0), 8)
    cv2.imwrite(os.path.join(img_dir, f"{n}.png"), img)

    yes_idx = random.sample(range(5), random.randint(1, 2))
    answers = []
    for k in range(5):
        img_k = np.ones((img_size, img_size, 3), np.uint8) * 255
        label = ""
        if k in yes_idx:
            draw_segments(img_k, main_segs, grid_size, img_size, margin, (0,0,0), 8)
            noise = add_noise_segments(all_segments, main_segs, n_noise=random.randint(2,4))
            draw_segments(img_k, noise, grid_size, img_size, margin, (0,0,0), 8)
            label = "yes"
        else:
            main_segs_no = erase_main_line(img_k, main_segs, grid_size, img_size, margin) or main_segs
            draw_segments(img_k, main_segs_no, grid_size, img_size, margin, (0,0,0), 8)
            noise = add_noise_segments(all_segments, main_segs_no, n_noise=random.randint(2,4))
            draw_segments(img_k, noise, grid_size, img_size, margin, (0,0,0), 8)
            label = "no"
        cv2.imwrite(os.path.join(img_dir, f"{n}-{k}.png"), img_k)
        answers.append(f"{n}-{k} {label}")
    return answers

if __name__ == "__main__":
    num_imgs = int(input("请输入要生成的主图数量："))
    img_dir = os.path.join(os.getcwd(), "Images")
    os.makedirs(img_dir, exist_ok=True)
    all_segments = get_all_grid_segments(3)
    all_answers = []
    for n in range(1, num_imgs+1):
        ans = generate_one(n, all_segments, img_dir)
        all_answers.extend(ans)
    with open(os.path.join(img_dir, "ans.txt"), "w", encoding="utf-8") as f:
        for line in all_answers:
            f.write(line+'\n')
    print("全部图片和ans.txt已生成到Images文件夹。")

全部图片和ans.txt已生成到Images文件夹。
